# Maven Ski Shop — Black Friday 2021

In [118]:
import openpyxl as xl
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pprint import pprint
from collections import defaultdict

# 1. Load Data

In [119]:
FILENAME = pd.read_excel("/content/maven_ski_shop_data.xlsx")

In [120]:
wb = xl.load_workbook(filename="/content/maven_ski_shop_data.xlsx")

In [121]:
order_ws = wb["Orders_Info"]
items_ws   = wb["Item_Info"]
inv_ws     = wb["Inventory_Levels"]

In [122]:
# Item lookup: {product_id: {name, price, cost}}
item_catalog = {
    row[0]: {"name": row[1], "price": row[2], "cost": row[3]}
    for row in items_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}

In [123]:
# Inventory lookup: {product_id: qty_in_stock}
inventory = {
    row[0]: row[1]
    for row in inv_ws.iter_rows(min_row=2, values_only=True)
    if row[0] is not None
}

In [124]:
item_catalog

{10001: {'name': 'Coffee', 'price': 5.99, 'cost': 0.99},
 10002: {'name': 'Beanie', 'price': 9.99, 'cost': 4.29},
 10003: {'name': 'Gloves', 'price': 19.99, 'cost': 7.99},
 10004: {'name': 'Sweatshirt', 'price': 24.99, 'cost': 10.59},
 10005: {'name': 'Helmet', 'price': 99.99, 'cost': 49.99},
 10006: {'name': 'Snow Pants', 'price': 79.99, 'cost': 32.49},
 10007: {'name': 'Coat', 'price': 119.99, 'cost': 54.55},
 10008: {'name': 'Ski Poles', 'price': 99.99, 'cost': 69.99},
 10009: {'name': 'Ski Boots', 'price': 199.99, 'cost': 89.99},
 10010: {'name': 'Skis', 'price': 599.99, 'cost': 249.99},
 10011: {'name': 'Snowboard Boots', 'price': 129.99, 'cost': 64.99},
 10012: {'name': 'Bindings', 'price': 149.99, 'cost': 89.99},
 10013: {'name': 'Snowboard', 'price': 499.99, 'cost': 199.99}}

In [125]:
inventory

{10001: 100,
 10002: 15,
 10003: 10,
 10004: 25,
 10005: 8,
 10006: 6,
 10007: 0,
 10008: 0,
 10009: 1,
 10010: 5,
 10011: 0,
 10012: 4}

# 2. TAX CALCULATOR

In [126]:
TAX_RATES = {
    "Sun Valley": 0.08,
    "Mammoth":    0.0775,
    "Stowe":      0.06,
}

def tax_calculator(subtotal: float, rate: float) -> tuple:
    """Returns (subtotal, tax_amount, total)."""
    tax = round(subtotal * rate, 2)
    total = round(subtotal + tax, 2)
    return (subtotal, tax, total)

In [127]:
print(tax_calculator(100, TAX_RATES["Sun Valley"]))

(100, 8.0, 108.0)


# 3. BUILD ORDER DICTIONARY

In [128]:
def build_order_dict(sheet) -> dict:
    """
    Keys:   Order_ID
    Values: [customer_id, date, subtotal, tax, total, location, items_list]
    """
    order_dict = {}
    for row in sheet.iter_rows(min_row=2, values_only=True):
        order_id, cust_id, date, subtotal, _, _, location, items_raw = row[:8]
        if order_id is None:
            continue

        # Parse items into a list of ints
        if isinstance(items_raw, str):
            item_list = [int(x.strip()) for x in items_raw.split(",")]
        else:
            item_list = [int(items_raw)]

        # Apply tax
        rate = TAX_RATES.get(location, 0)
        _, tax, total = tax_calculator(subtotal, rate)

        order_dict[order_id] = [cust_id, date, subtotal, tax, total, location, item_list]

    return order_dict


order_dict = build_order_dict(order_ws)

# Named indices for readability
CUSTOMER_ID, DATE, SUBTOTAL, TAX, TOTAL, LOCATION, ITEMS = 0, 1, 2, 3, 4, 5, 6

# 4. HELPER FUNCTIONS

In [129]:
def column_printer(sheet, column: str):
    """Print all values in a given column of a worksheet."""
    for cell in sheet[column]:
        print(f"  {cell.coordinate}: {cell.value}")


def column_sum(column_index: int, dictionary: dict) -> float:
    """Sum all values at a given index across dictionary values."""
    return round(sum(v[column_index] for v in dictionary.values()), 2)


def aggregator(group_index: int, sum_index: int, dictionary: dict) -> dict:
    """
    Group by unique values at group_index, sum values at sum_index.
    Example: aggregator(LOCATION, SUBTOTAL, order_dict)
    """
    result = defaultdict(float)
    for data in dictionary.values():
        result[data[group_index]] += data[sum_index]
    return {k: round(v, 2) for k, v in result.items()}

# 5. ANALYSIS SUMMARY

In [130]:
def print_analysis():
    print("\n" + "=" * 50)
    print("  MAVEN SKI SHOP — BLACK FRIDAY ANALYSIS")
    print("=" * 50)

    total_subtotal = column_sum(SUBTOTAL, order_dict)
    total_tax      = column_sum(TAX,      order_dict)
    total_revenue  = column_sum(TOTAL,    order_dict)
    avg_order      = round(total_subtotal / len(order_dict), 2)
    unique_custs   = len(set(v[CUSTOMER_ID] for v in order_dict.values()))
    opc            = round(len(order_dict) / unique_custs, 2)
    total_items    = sum(len(v[ITEMS]) for v in order_dict.values())

    print(f"\n  📦 Total orders:          {len(order_dict)}")
    print(f"  👤 Unique customers:       {unique_custs}")
    print(f"  🛒 Orders per customer:    {opc}")
    print(f"  🎿 Total items sold:       {total_items}")
    print(f"\n  💰 Total subtotal:        ${total_subtotal:,.2f}")
    print(f"  🏷  Total tax collected:   ${total_tax:,.2f}")
    print(f"  💵 Total revenue:          ${total_revenue:,.2f}")
    print(f"  📊 Avg order value:        ${avg_order:,.2f}")

    print("\n  📍 Revenue by Location:")
    for loc, rev in sorted(aggregator(LOCATION, SUBTOTAL, order_dict).items(),
                           key=lambda x: -x[1]):
        print(f"     {loc:<15} ${rev:>9,.2f}")

    print("\n  📅 Revenue by Date:")
    for date, rev in sorted(aggregator(DATE, SUBTOTAL, order_dict).items()):
        print(f"     {date}   ${rev:>9,.2f}")

    print("\n  🏆 Top 5 Products by Units Sold:")
    prod_units = defaultdict(int)
    for v in order_dict.values():
        for pid in v[ITEMS]:
            name = item_catalog.get(pid, {}).get("name", f"#{pid}")
            prod_units[name] += 1
    for name, qty in sorted(prod_units.items(), key=lambda x: -x[1])[:5]:
        print(f"     {name:<20} {qty} units")

    print("\n  📦 Inventory Alerts:")
    for pid, qty in inventory.items():
        name = item_catalog.get(pid, {}).get("name", f"#{pid}")
        if qty == 0:
            print(f"     ⛔ {name:<20} OUT OF STOCK")
        elif qty <= 5:
            print(f"     ⚠️  {name:<20} Only {qty} left")

    print("\n" + "=" * 50 + "\n")

# 6. INTERACTIVE PLOTLY CHARTS

In [131]:
def build_dashboard(location_filter="All"):
    """Build a multi-panel Plotly dashboard and save as HTML."""

    current_order_dict = order_dict
    if location_filter != "All":
        current_order_dict = {
            oid: data for oid, data in order_dict.items()
            if data[LOCATION] == location_filter
        }

    orders_df = pd.DataFrame([
        {
            "order_id":    oid,
            "customer_id": v[CUSTOMER_ID],
            "date":        v[DATE],
            "subtotal":    v[SUBTOTAL],
            "tax":         v[TAX],
            "total":       v[TOTAL],
            "location":    v[LOCATION],
            "num_items":   len(v[ITEMS]),
        }
        for oid, v in current_order_dict.items()
    ])

    # Product unit counts
    prod_units = defaultdict(int)
    for v in current_order_dict.values():
        for pid in v[ITEMS]:
            prod_units[item_catalog.get(pid, {}).get("name", f"#{pid}")] += 1
    prod_df = pd.DataFrame(list(prod_units.items()), columns=["product", "units_sold"])\
                .sort_values("units_sold", ascending=False)

    # Inventory df (Note: inventory is global, not filtered by location in this context)
    inv_df = pd.DataFrame([
        {"product": item_catalog.get(pid, {}).get("name", f"#{pid}"), "qty": qty}
        for pid, qty in inventory.items()
    ])

    loc_colors = {
        "Sun Valley": "#378ADD",
        "Mammoth":    "#1D9E75",
        "Stowe":      "#7F77DD",
    }

    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            "Revenue by Location (Subtotal)",
            "Daily Revenue",
            "Units Sold by Product",
            "Orders per Day by Location",
            "Top Products Revenue",
            "Inventory Levels",
        ),
        specs=[
            [{"type": "pie"},     {"type": "bar"}],
            [{"type": "bar"},     {"type": "bar"}],
            [{"type": "bar"},     {"type": "bar"}],
        ],
        vertical_spacing=0.12,
        horizontal_spacing=0.1,
    )

    # ── Chart 1: Pie — revenue by location
    loc_rev = aggregator(LOCATION, SUBTOTAL, current_order_dict)
    fig.add_trace(go.Pie(
        labels=list(loc_rev.keys()),
        values=list(loc_rev.values()),
        marker_colors=[loc_colors[l] for l in loc_rev.keys()],
        textinfo="label+percent",
        hovertemplate="%{label}: $%{value:,.2f}<extra></extra>",
    ), row=1, col=1)

    # ── Chart 2: Bar — daily revenue
    date_rev = aggregator(DATE, SUBTOTAL, current_order_dict)
    dates = sorted(date_rev.keys())
    date_labels = [f"Nov {d.split('/')[1]}" for d in dates]
    fig.add_trace(go.Bar(
        x=date_labels,
        y=[date_rev[d] for d in dates],
        marker_color="#378ADD",
        hovertemplate="$%{y:,.2f}<extra></extra>",
    ), row=1, col=2)

    # ── Chart 3: Horizontal bar — units sold
    fig.add_trace(go.Bar(
        x=prod_df["units_sold"],
        y=prod_df["product"],
        orientation="h",
        marker_color="#1D9E75",
        hovertemplate="%{y}: %{x} units<extra></extra>",
    ), row=2, col=1)

    # ── Chart 4: Stacked bar — orders by date & location
    loc_date_orders = orders_df.groupby(["date", "location"]).size().unstack(fill_value=0)
    for loc in loc_date_orders.columns:
        fig.add_trace(go.Bar(
            name=loc,
            x=[f"Nov {d.split('/')[1]}" for d in sorted(loc_date_orders.index)],
            y=[loc_date_orders.loc[d, loc] for d in sorted(loc_date_orders.index)],
            marker_color=loc_colors.get(loc, "#888"),
            hovertemplate=f"{loc}: %{{y}} orders<extra></extra>",
        ), row=2, col=2)

    # ── Chart 5: Horizontal bar — product revenue
    prod_rev = defaultdict(float)
    for v in current_order_dict.values():
        for pid in v[ITEMS]:
            name = item_catalog.get(pid, {}).get("name", f"#{pid}")
            prod_rev[name] += item_catalog.get(pid, {}).get("price", 0)
    prod_rev_df = pd.DataFrame(list(prod_rev.items()), columns=["product", "revenue"])\
                    .sort_values("revenue", ascending=True).tail(8)
    fig.add_trace(go.Bar(
        x=prod_rev_df["revenue"],
        y=prod_rev_df["product"],
        orientation="h",
        marker_color="#7F77DD",
        hovertemplate="$%{x:,.2f}<extra></extra>",
    ), row=3, col=1)

    # ── Chart 6: Inventory levels (color coded)
    inv_colors = ["#E24B4A" if q == 0 else "#EF9F27" if q <= 5 else "#1D9E75"
                  for q in inv_df["qty"]]
    fig.add_trace(go.Bar(
        x=inv_df["qty"],
        y=inv_df["product"],
        orientation="h",
        marker_color=inv_colors,
        hovertemplate="%{y}: %{x} in stock<extra></extra>",
    ), row=3, col=2)

    fig.update_layout(
        title_text="Maven Ski Shop — Black Friday 2021 Dashboard",
        title_font_size=20,
        height=1100,
        showlegend=False,
        barmode="stack",
        paper_bgcolor="#ffffff",
        plot_bgcolor="#f8f9fb",
        font=dict(family="Arial, sans-serif", size=12, color="#333"),
    )
    fig.update_xaxes(showgrid=True, gridcolor="#e5e7eb")
    fig.update_yaxes(showgrid=False)

    out = "maven_ski_shop_dashboard.html"
    fig.write_html(out, include_plotlyjs="cdn")
    # print(f"  ✅ Dashboard saved → {out}") # Commented out to avoid re-printing on every widget update
    # print("     Open it in any browser for full interactivity.\n") # Commented out

    # Also show inline if in a Jupyter/IPython context
    try:
        get_ipython
        fig.show()
    except NameError:
        pass

In [132]:
def export_dashboard_data_to_excel():
    """Exports the raw data used in the dashboard to an Excel file."""

    # Re-create dataframes similar to build_dashboard but without filtering
    orders_df = pd.DataFrame([
        {
            "order_id":    oid,
            "customer_id": v[CUSTOMER_ID],
            "date":        v[DATE],
            "subtotal":    v[SUBTOTAL],
            "tax":         v[TAX],
            "total":       v[TOTAL],
            "location":    v[LOCATION],
            "num_items":   len(v[ITEMS]),
        }
        for oid, v in order_dict.items()
    ])

    prod_units = defaultdict(int)
    for v in order_dict.values():
        for pid in v[ITEMS]:
            prod_units[item_catalog.get(pid, {}).get("name", f"#{pid}")] += 1
    prod_df = pd.DataFrame(list(prod_units.items()), columns=["product", "units_sold"])\
                .sort_values("units_sold", ascending=False)

    inv_df = pd.DataFrame([
        {"product": item_catalog.get(pid, {}).get("name", f"#{pid}"), "qty": qty}
        for pid, qty in inventory.items()
    ])

    output_filename = "maven_ski_shop_dashboard_data.xlsx"
    with pd.ExcelWriter(output_filename) as writer:
        orders_df.to_excel(writer, sheet_name='Orders Data', index=False)
        prod_df.to_excel(writer, sheet_name='Products Sold', index=False)
        inv_df.to_excel(writer, sheet_name='Inventory Levels', index=False)
    print(f"  ✅ Dashboard data exported to {output_filename}")

export_dashboard_data_to_excel()

  ✅ Dashboard data exported to maven_ski_shop_dashboard_data.xlsx


### Generate Excel Report with Embedded Static Charts

This code will create an Excel workbook with multiple sheets:
- **Raw Data Sheets**: Contains the `Orders`, `Products Sold`, and `Inventory Levels` data.
- **Chart Sheets**: Each sheet will contain a static `Matplotlib` chart visually representing aspects of the dashboard, such as revenue by location, daily revenue, and product sales. This provides a visual summary similar to the Plotly dashboard within an Excel file.

In [133]:
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.drawing.image import Image as ExcelImage

def generate_excel_report_with_charts():
    """Generates an Excel report with data and embedded static charts."""
    wb = Workbook()

    # Remove default sheet
    if 'Sheet' in wb.sheetnames:
        wb.remove(wb['Sheet'])

    # 1. Prepare DataFrames (similar to build_dashboard)
    orders_df = pd.DataFrame([
        {
            "order_id": oid,
            "customer_id": v[CUSTOMER_ID],
            "date": v[DATE],
            "subtotal": v[SUBTOTAL],
            "tax": v[TAX],
            "total": v[TOTAL],
            "location": v[LOCATION],
            "num_items": len(v[ITEMS]),
        }
        for oid, v in order_dict.items()
    ])

    prod_units = defaultdict(int)
    for v in order_dict.values():
        for pid in v[ITEMS]:
            prod_units[item_catalog.get(pid, {}).get("name", f"#{pid}")] += 1
    prod_df = pd.DataFrame(list(prod_units.items()), columns=["product", "units_sold"])\
                .sort_values("units_sold", ascending=False)

    inv_df = pd.DataFrame([
        {"product": item_catalog.get(pid, {}).get("name", f"#{pid}"), "qty": qty}
        for pid, qty in inventory.items()
    ])

    # 2. Add Raw Data Sheets
    with pd.ExcelWriter('maven_ski_shop_excel_report.xlsx', engine='openpyxl') as writer:
        # Write DataFrames to sheets
        orders_df.to_excel(writer, sheet_name='Orders Data', index=False)
        prod_df.to_excel(writer, sheet_name='Products Sold', index=False)
        inv_df.to_excel(writer, sheet_name='Inventory Levels', index=False)

        # Access the workbook from the writer
        wb = writer.book

        # 3. Generate and Embed Charts

        # Chart 1: Revenue by Location (Pie Chart)
        loc_rev = aggregator(LOCATION, SUBTOTAL, order_dict)
        fig1, ax1 = plt.subplots(figsize=(8, 6))
        ax1.pie(loc_rev.values(), labels=loc_rev.keys(), autopct='%1.1f%%', startangle=90)
        ax1.set_title('Revenue by Location (Subtotal)')
        ax1.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
        chart_path1 = 'revenue_by_location.png'
        fig1.savefig(chart_path1)
        plt.close(fig1)
        ws1 = wb.create_sheet('Revenue by Location')
        ws1.add_image(ExcelImage(chart_path1), 'A1')

        # Chart 2: Daily Revenue (Bar Chart)
        date_rev = aggregator(DATE, SUBTOTAL, order_dict)
        dates = sorted(date_rev.keys())
        revenues = [date_rev[d] for d in dates]
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        ax2.bar([f"Nov {d.split('/')[1]}" for d in dates], revenues, color='#378ADD')
        ax2.set_title('Daily Revenue')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Total Revenue ($)')
        ax2.tick_params(axis='x', rotation=45)
        fig2.tight_layout()
        chart_path2 = 'daily_revenue.png'
        fig2.savefig(chart_path2)
        plt.close(fig2)
        ws2 = wb.create_sheet('Daily Revenue')
        ws2.add_image(ExcelImage(chart_path2), 'A1')

        # Chart 3: Units Sold by Product (Horizontal Bar Chart)
        prod_df_sorted = prod_df.sort_values("units_sold", ascending=True).tail(10)
        fig3, ax3 = plt.subplots(figsize=(10, 7))
        ax3.barh(prod_df_sorted["product"], prod_df_sorted["units_sold"], color='#1D9E75')
        ax3.set_title('Units Sold by Product')
        ax3.set_xlabel('Units Sold')
        ax3.set_ylabel('Product')
        fig3.tight_layout()
        chart_path3 = 'units_sold_by_product.png'
        fig3.savefig(chart_path3)
        plt.close(fig3)
        ws3 = wb.create_sheet('Units Sold')
        ws3.add_image(ExcelImage(chart_path3), 'A1')

        # Chart 4: Inventory Levels (Horizontal Bar Chart)
        inv_df_sorted = inv_df.sort_values("qty", ascending=True)
        colors = ["red" if q == 0 else "orange" if q <= 5 else "green" for q in inv_df_sorted["qty"]]
        fig4, ax4 = plt.subplots(figsize=(10, 7))
        ax4.barh(inv_df_sorted["product"], inv_df_sorted["qty"], color=colors)
        ax4.set_title('Inventory Levels')
        ax4.set_xlabel('Quantity in Stock')
        ax4.set_ylabel('Product')
        fig4.tight_layout()
        chart_path4 = 'inventory_levels.png'
        fig4.savefig(chart_path4)
        plt.close(fig4)
        ws4 = wb.create_sheet('Inventory Levels')
        ws4.add_image(ExcelImage(chart_path4), 'A1')

    print(f"  ✅ Excel report with charts saved → maven_ski_shop_excel_report.xlsx")

generate_excel_report_with_charts()

  ✅ Excel report with charts saved → maven_ski_shop_excel_report.xlsx


# 7. SAVE FIXED WORKBOOK

In [134]:
def save_fixed_workbook():
    """Write calculated tax & total back into the workbook and save."""
    for index, order in enumerate(order_dict.values(), start=2):
        order_ws[f"E{index}"] = order[TAX]
        order_ws[f"F{index}"] = order[TOTAL]
    wb.save("maven_ski_shop_data_fixed.xlsx")
    print("  ✅ Fixed workbook saved → maven_ski_shop_data_fixed.xlsx\n")

# 8. INTERACTIVE MENU

In [135]:
MENU = """
╔══════════════════════════════════════╗
║   Maven Ski Shop — Interactive CLI   ║
╠══════════════════════════════════════╣
║  1. Print full analysis summary      ║
║  2. Revenue by location              ║
║  3. Revenue by date                  ║
║  4. Revenue by customer              ║
║  5. Product units sold               ║
║  6. Inventory status                 ║
║  7. 🌐 Open Plotly dashboard (HTML)  ║
║  8. 💾 Save fixed workbook           ║
║  9. Explore a specific order         ║
║  0. Exit                             ║
╚══════════════════════════════════════╝
"""

def run_menu():
    while True:
        print(MENU)
        choice = input("  Enter option: ").strip()

        if choice == "1":
            print_analysis()

        elif choice == "2":
            print("\n  📍 Revenue by Location (subtotal):")
            for loc, rev in sorted(aggregator(LOCATION, SUBTOTAL, order_dict).items(),
                                   key=lambda x: -x[1]):
                pct = rev / column_sum(SUBTOTAL, order_dict) * 100
                bar = "█" * int(pct / 2)
                print(f"  {loc:<15} ${rev:>9,.2f}  {pct:5.1f}%  {bar}")
            input("\n  [Enter to continue]")

        elif choice == "3":
            print("\n  📅 Revenue by Date:")
            for date, rev in sorted(aggregator(DATE, SUBTOTAL, order_dict).items()):
                count = sum(1 for v in order_dict.values() if v[DATE] == date)
                print(f"  {date}  ${rev:>9,.2f}  ({count} orders)")
            input("\n  [Enter to continue]")

        elif choice == "4":
            print("\n  👤 Revenue by Customer (top 10):")
            cust_rev = aggregator(CUSTOMER_ID, TOTAL, order_dict)
            for cid, rev in sorted(cust_rev.items(), key=lambda x: -x[1])[:10]:
                print(f"  {cid:<10}  ${rev:>9,.2f}")
            input("\n  [Enter to continue]")

        elif choice == "5":
            print("\n  🎿 Units Sold by Product:")
            prod_units = defaultdict(int)
            for v in order_dict.values():
                for pid in v[ITEMS]:
                    name = item_catalog.get(pid, {}).get("name", f"#{pid}")
                    prod_units[name] += 1
            for name, qty in sorted(prod_units.items(), key=lambda x: -x[1]):
                bar = "■" * qty
                print(f"  {name:<20} {qty:>3}  {bar}")
            input("\n  [Enter to continue]")

        elif choice == "6":
            print("\n  📦 Inventory Status:")
            for pid, qty in sorted(inventory.items()):
                name = item_catalog.get(pid, {}).get("name", f"#{pid}")
                if qty == 0:
                    status = "⛔ OUT OF STOCK"
                elif qty <= 5:
                    status = f"⚠️  Low ({qty})"
                else:
                    status = f"✅ OK ({qty})"
                print(f"  {name:<22} {status}")
            input("\n  [Enter to continue]")

        elif choice == "7":
            print("\n  Building dashboard...")
            build_dashboard()
            import webbrowser, os
            webbrowser.open("file://" + os.path.abspath("maven_ski_shop_dashboard.html"))
            input("  [Enter to continue]")

        elif choice == "8":
            save_fixed_workbook()
            input("  [Enter to continue]")

        elif choice == "9":
            oid_input = input("  Enter Order ID (e.g. 100000): ").strip()
            try:
                oid = int(oid_input)
                if oid in order_dict:
                    v = order_dict[oid]
                    print(f"\n  Order #{oid}")
                    print(f"  Customer:  {v[CUSTOMER_ID]}")
                    print(f"  Date:      {v[DATE]}")
                    print(f"  Location:  {v[LOCATION]}")
                    print(f"  Subtotal:  ${v[SUBTOTAL]:.2f}")
                    print(f"  Tax:       ${v[TAX]:.2f}")
                    print(f"  Total:     ${v[TOTAL]:.2f}")
                    print(f"  Items:     {', '.join(item_catalog.get(p, {}).get('name', f'#{p}') for p in v[ITEMS])}")
                else:
                    print(f"  Order {oid} not found.")
            except ValueError:
                print("  Invalid order ID.")
            input("\n  [Enter to continue]")

        elif choice == "0":
            print("\n  Goodbye! ⛷\n")
            break
        else:
            print("  Invalid option, try again.")

In [136]:
build_dashboard()

In [137]:
run_menu()


╔══════════════════════════════════════╗
║   Maven Ski Shop — Interactive CLI   ║
╠══════════════════════════════════════╣
║  1. Print full analysis summary      ║
║  2. Revenue by location              ║
║  3. Revenue by date                  ║
║  4. Revenue by customer              ║
║  5. Product units sold               ║
║  6. Inventory status                 ║
║  7. 🌐 Open Plotly dashboard (HTML)  ║
║  8. 💾 Save fixed workbook           ║
║  9. Explore a specific order         ║
║  0. Exit                             ║
╚══════════════════════════════════════╝



KeyboardInterrupt: Interrupted by user

### Interactive Dashboard Displaying Inventory and Product Sales

Below is the interactive dashboard, which includes visualizations for:

*   **Units Sold by Product:** This chart helps identify the most and least popular products.
*   **Inventory Levels:** This chart provides a clear overview of current stock, highlighting items that are out of stock or running low.

In [138]:
build_dashboard()

### Detailed Inventory and Sales Summary

Below is a direct printout of the analysis, including units sold by product and current inventory levels.

In [139]:
print_analysis()


  MAVEN SKI SHOP — BLACK FRIDAY ANALYSIS

  📦 Total orders:          27
  👤 Unique customers:       19
  🛒 Orders per customer:    1.42
  🎿 Total items sold:       54

  💰 Total subtotal:        $8,731.47
  🏷  Total tax collected:   $617.20
  💵 Total revenue:          $9,348.67
  📊 Avg order value:        $323.39

  📍 Revenue by Location:
     Mammoth         $ 3,879.81
     Stowe           $ 3,582.82
     Sun Valley      $ 1,268.84

  📅 Revenue by Date:
     11/26/2021   $ 5,515.65
     11/27/2021   $ 1,600.92
     11/28/2021   $ 1,614.90

  🏆 Top 5 Products by Units Sold:
     Ski Poles            6 units
     Ski Boots            6 units
     Skis                 6 units
     Beanie               5 units
     Helmet               5 units

  📦 Inventory Alerts:
     ⛔ Coat                 OUT OF STOCK
     ⛔ Ski Poles            OUT OF STOCK
     ⚠️  Ski Boots            Only 1 left
     ⚠️  Skis                 Only 5 left
     ⛔ Snowboard Boots      OUT OF STOCK
     ⚠️  Bindings 